# JMJ Hitting Analysis — Phase 3 Pipeline (Colab)

**목적**: 로컬 PC 자원이 부족한 경우 Google Colab에서 Phase 3(채널 판별 → OCR 구간 추출 → 스윙 탐지)를 실행합니다.

**순서**
1. Google Drive 마운트
2. 저장소 클론 & 패키지 설치
3. 분석할 영상 선택
4. Phase 3 실행
5. 결과 확인 & Drive에 저장

## Step 1 — Google Drive 마운트

Drive에 영상 파일과 결과물을 저장하기 위해 마운트합니다.  
실행 후 팝업에서 계정을 선택하고 권한을 허용해 주세요.

In [4]:
from google.colab import drive
drive.mount('/content/drive')

# Drive 안에 작업 폴더 경로 (필요시 수정)
DRIVE_ROOT = '/content/drive/MyDrive/JMJ_Hitting_Analysis'

import os
os.makedirs(f'{DRIVE_ROOT}/data', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/outputs/json', exist_ok=True)
print('Drive 마운트 완료:', DRIVE_ROOT)

ModuleNotFoundError: No module named 'google.colab'

## Step 2 — 저장소 클론 & 패키지 설치

처음 실행하거나 코드를 업데이트할 때마다 실행하세요.  
이미 클론되어 있으면 `git pull`만 수행합니다.

In [ ]:
import os

REPO_DIR = '/content/JMJ_Hitting_Analysis'
REPO_URL = 'https://github.com/JoeYunHa/JMJ_Hitting_Analysis.git'  # ← 본인 GitHub URL로 교체

if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
    !git pull
else:
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

print('작업 디렉토리:', os.getcwd())

In [ ]:
# PaddlePaddle GPU 버전 설치 (CUDA 11.x 기준 — Colab 기본 환경)
!pip install paddlepaddle-gpu -i https://pypi.tuna.tsinghua.edu.cn/simple -q
!pip install paddleocr -q
!pip install opencv-python-headless numpy tqdm python-dotenv -q

# 설치 확인
import paddle
print('PaddlePaddle version:', paddle.__version__)
print('GPU available:', paddle.is_compiled_with_cuda())

## Step 3 — 분석할 영상 선택

아래 두 가지 방법 중 하나를 선택하세요.

| 방법 | 설명 | 권장 상황 |
|------|------|----------|
| **A. Drive에서 복사** | Drive에 미리 업로드한 영상을 Colab 로컬로 복사 | 영상을 재사용하는 경우 |
| **B. yt-dlp 직접 다운로드** | YouTube URL에서 Colab 로컬로 직접 다운로드 | 빠른 1회성 테스트 |

In [ ]:
# 방법 A: Drive에서 Colab 로컬(/content/data/)로 복사
# Drive에 업로드된 영상 파일명을 아래에 입력하세요.

VIDEO_FILENAME = '20260328_[롯데자이언츠 vs 삼성라이온즈] 3.28(토) 야구 하이라이트｜2026 KBO 리그｜KB.webm'

DRIVE_VIDEO = f'{DRIVE_ROOT}/data/{VIDEO_FILENAME}'
LOCAL_VIDEO = f'/content/data/{VIDEO_FILENAME}'

os.makedirs('/content/data', exist_ok=True)

if not os.path.exists(LOCAL_VIDEO):
    print('Drive → Colab 로컬로 복사 중...')
    import shutil
    shutil.copy2(DRIVE_VIDEO, LOCAL_VIDEO)
    print('복사 완료:', LOCAL_VIDEO)
else:
    print('이미 존재합니다:', LOCAL_VIDEO)

VIDEO_PATH = LOCAL_VIDEO

In [ ]:
# 방법 B: yt-dlp로 YouTube에서 직접 다운로드
# 위 방법 A 셀을 실행했다면 이 셀은 건너뛰세요.

!pip install yt-dlp -q

YOUTUBE_URL = 'https://www.youtube.com/watch?v=XXXXXXXXXX'  # ← 실제 URL로 교체
os.makedirs('/content/data', exist_ok=True)

!yt-dlp -f 'bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]' \
    --output '/content/data/%(upload_date)s_%(title)s.%(ext)s' \
    {YOUTUBE_URL}

# 다운로드된 파일 확인
import glob
files = glob.glob('/content/data/*')
print('다운로드된 파일:', files)
VIDEO_PATH = files[0] if files else ''

## Step 4 — Phase 3 실행

채널 판별 → OCR 구간 추출 → 스윙 탐지를 순서대로 실행합니다.

In [ ]:
# 환경 설정 (API 키는 Phase 3에서는 불필요하지만 settings.py 임포트를 위해 설정)
import os
os.environ['YOUTUBE_API_KEY'] = 'dummy_not_needed_for_phase3'

# 저장소 루트를 Python 경로에 추가
import sys
sys.path.insert(0, '/content/JMJ_Hitting_Analysis')
os.chdir('/content/JMJ_Hitting_Analysis')

print('VIDEO_PATH:', VIDEO_PATH)
print('sys.path[0]:', sys.path[0])

In [ ]:
# Step 4-1: 채널 판별
from src.detection.channel_detector import load_all_configs, detect_channel_from_video

configs = load_all_configs()
print(f'로드된 채널 config 수: {len(configs)}')
print('등록된 config:', list(configs.keys()))

config_id = detect_channel_from_video(VIDEO_PATH, configs)
if config_id:
    print(f'\n✅ 감지된 채널: {config_id}')
    cfg = configs[config_id]
else:
    print('\n❌ 채널 판별 실패 — 로고 템플릿 경로나 threshold를 확인하세요.')

In [ ]:
# Step 4-2: OCR 구간 추출 (가장 오래 걸리는 단계)
from src.detection.ocr_extractor import extract_batter_segments

TARGET_NAME = '전민재'

print(f'OCR 구간 추출 시작 — 타겟: {TARGET_NAME}')
print('(영상 길이에 따라 수 분이 소요될 수 있습니다)')

segments = extract_batter_segments(
    VIDEO_PATH,
    cfg,
    target_name=TARGET_NAME,
    sample_fps=2,
    ocr_delay_sec=3.0,
)

print(f'\n✅ 추출된 구간 수: {len(segments)}')
for i, seg in enumerate(segments):
    print(f'  [{i+1}] {seg.start_sec:.1f}s ~ {seg.end_sec:.1f}s  '
          f'({(seg.end_sec - seg.start_sec):.1f}s)')

In [ ]:
# Step 4-3: 스윙 탐지
from src.detection.swing_detector import _detect_swings, FlowConfig
from src.utils.video import open_video

print('스윙 탐지 시작...')
with open_video(VIDEO_PATH) as cap:
    for seg in segments:
        seg.swing_frames = _detect_swings(
            cap, seg.start_frame, seg.end_frame,
            flow_threshold=4.0, cooldown_frames=15, flow_config=FlowConfig(),
        )
        print(f'  {seg.start_sec:.1f}s ~ {seg.end_sec:.1f}s → 스윙 {len(seg.swing_frames)}회')

total_swings = sum(len(s.swing_frames) for s in segments)
print(f'\n✅ 전체 스윙 감지 수: {total_swings}')

## Step 5 — 결과 저장

JSON을 Colab 로컬과 Google Drive 양쪽에 저장합니다.

In [ ]:
import json
from pathlib import Path

# Colab 로컬 저장
LOCAL_OUT_DIR = '/content/outputs/json'
os.makedirs(LOCAL_OUT_DIR, exist_ok=True)

stem = Path(VIDEO_PATH).stem
out_path = f'{LOCAL_OUT_DIR}/{stem}_segments.json'

result = {
    'channel': cfg['channel'],
    'config_id': config_id,
    'target': TARGET_NAME,
    'segments': [seg.to_dict() for seg in segments],
}

with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print(f'로컬 저장 완료: {out_path}')

# Drive로 복사
import shutil
drive_out = f'{DRIVE_ROOT}/outputs/json/{stem}_segments.json'
shutil.copy2(out_path, drive_out)
print(f'Drive 저장 완료: {drive_out}')

In [ ]:
# 결과 미리보기
print(json.dumps(result, ensure_ascii=False, indent=2)[:2000])

## (선택) 전체 파이프라인을 한 번에 실행

위의 단계별 셀 대신 `run_phase3()`를 직접 호출하는 방법입니다.

In [ ]:
# 환경 변수·경로 설정이 완료된 상태에서 실행하세요 (Step 4-0 셀 필수)
from src.detection.segment_pipeline import run_phase3

out_json = run_phase3(
    video_path=VIDEO_PATH,
    output_dir='/content/outputs/json',
    target_name='전민재',
)
print('결과 파일:', out_json)

# Drive에도 복사
import shutil
from pathlib import Path
shutil.copy2(out_json, f"{DRIVE_ROOT}/outputs/json/{Path(out_json).name}")
print('Drive 저장 완료')